# 5.3 混合并行策略 (Hybrid Parallelism)

实际大规模训练中，单一并行方式无法满足需求，需要组合多种并行策略。3D并行（DP+TP+PP）是训练GPT-3、PaLM、LLaMA等超大模型的标准方案。

本节涵盖：
- 3D并行布局与GPU拓扑
- 1F1B流水线调度
- 通信-计算重叠
- 并行配置搜索
- ZeRO与3D并行的结合

## 1. 3D并行布局与GPU拓扑

**目的**：理解3D并行的GPU布局原则和通信模式

**核心原则**：
- **TP在节点内**：利用NVLink 600GB/s高带宽，AllReduce延迟低
- **PP跨节点**：利用InfiniBand 200Gbps带宽，点对点通信
- **DP全局**：跨所有节点，AllReduce梯度同步

**GPU布局示例（64 GPU = 8节点 × 8GPU）**：
```
TP=8, PP=4, DP=2
Node 0: GPU 0-7  → TP group 0, PP stage 0, DP rank 0
Node 1: GPU 8-15 → TP group 0, PP stage 1, DP rank 0
Node 2: GPU 16-23→ TP group 0, PP stage 2, DP rank 0
Node 3: GPU 24-31→ TP group 0, PP stage 3, DP rank 0
Node 4-7: DP rank 1 (replica of nodes 0-3)
```

**通信量分析**：
- TP AllReduce：每个Transformer层2次，数据量=2×batch×seq×d_model×4/TP_size
- PP P2P：每阶段1次发送+1次接收，数据量=batch×seq×d_model
- DP AllReduce：每训练步1次，数据量=模型参数量/DP_size

In [ ]:
import torch

print('=== 3D Parallelism Layout & Communication Analysis ===')

configs = [
    ('7B (LLaMA-7B)', 7, 32, 4096, 32, 2048, 1),
    ('70B (LLaMA-70B)', 70, 80, 8192, 64, 4096, 1),
    ('175B (GPT-3 scale)', 175, 96, 12288, 96, 4096, 1),
]

for name, params_b, n_layers, d_model, n_heads, seq_len, bs in configs:
    print(f'\n--- {name} ---')
    model_size_gb = params_b * 2
    print(f'Model size: {model_size_gb:.0f} GB (BF16)')

    tp_options = [1, 2, 4, 8]
    pp_options = [1, 2, 4, 8, 16]
    best = None
    for tp in tp_options:
        for pp in pp_options:
            per_gpu_model = model_size_gb / tp / pp
            optimizer_state = per_gpu_model * 6
            activation = d_model * n_layers * 4 * 2 * seq_len * bs / tp / 1e9
            total_per_gpu = per_gpu_model + optimizer_state + activation
            if total_per_gpu <= 80 and per_gpu_model <= 80:
                dp = 512 // (tp * pp)
                if dp >= 1:
                    tp_comm = 2 * bs * seq_len * d_model * 4 * 2 / tp / 1e9
                    pp_comm = bs * seq_len * d_model * 2 / 1e9
                    dp_comm = params_b * 1e9 * 4 / dp / 1e9
                    if best is None or dp > best[3]:
                        best = (tp, pp, total_per_gpu, dp, tp_comm, pp_comm, dp_comm)

    if best:
        tp, pp, gpu_mem, dp, tp_c, pp_c, dp_c = best
        print(f'  Recommended: TP={tp}, PP={pp}, DP={dp} (total={tp*pp*dp} GPUs)')
        print(f'  Per-GPU memory: {gpu_mem:.1f} GB')
        print(f'  TP communication: {tp_c:.2f} GB/layer (NVLink)')
        print(f'  PP communication: {pp_c:.2f} GB/stage (InfiniBand)')
        print(f'  DP communication: {dp_c:.0f} GB/step (AllReduce)')
    else:
        print(f'  Need ZeRO-3 or more GPUs')

print(f'\nKey: TP within node (NVLink), PP across nodes (InfiniBand), DP globally.')
print(f'Goal: maximize DP size (more data parallelism = higher throughput).')
print(f'Constraint: per-GPU memory must fit in 80GB (A100-80GB).')

## 2. 1F1B流水线调度

**目的**：理解流水线并行的核心调度算法，减少气泡（bubble）

**调度策略对比**：
- **GPipe**：先做所有前向（1F1F...1F），再做所有反向（1B1B...1B），气泡率=（PP-1）/PP
- **1F1B**：交替执行1个前向和1个反向，气泡率=（PP-1）/（PP+2×micro_batch-1），更优
- **Interleaved 1F1B**：每个设备负责多个不连续的stage，进一步减少气泡

**1F1B执行流程**（PP=4, micro_batch=8）：
```
Stage 0: F0 F1 F2 F3 B0 F4 B1 F5 B2 F6 B3 F7 B4 B5 B6 B7
Stage 1:    F0 F1 F2 B0 F3 B1 F4 B2 F5 B3 F6 B4 F7 B5 B6 B7
Stage 2:       F0 F1 B0 F2 B1 F3 B2 F4 B3 F5 B4 F6 B5 F7 B6 B7
Stage 3:          F0 B0 F1 B1 F2 B2 F3 B3 F4 B4 F5 B5 F6 B6 F7 B7
```

**峰值显存**：1F1B的峰值显存=1个micro_batch的激活值+PP-1个micro_batch的激活值

In [ ]:
import torch

print('=== Pipeline Schedule Comparison ===')

def gpipe_bubble_rate(pp, micro_batch):
    return (pp - 1) / (pp + micro_batch - 1)

def onefoneb_bubble_rate(pp, micro_batch):
    return (pp - 1) / (micro_batch + pp - 1)

def interleaved_bubble_rate(pp, micro_batch, n_chunks):
    return (pp - 1) / (micro_batch + pp - 1) / n_chunks

print(f'{"PP":>4} {"MB":>4} {"GPipe":>10} {"1F1B":>10} {"Interleaved":>12}')
for pp in [4, 8, 16]:
    for mb in [8, 16, 32, 64]:
        gpipe = gpipe_bubble_rate(pp, mb)
        ofob = onefoneb_bubble_rate(pp, mb)
        interleaved = interleaved_bubble_rate(pp, mb, 2)
        print(f'{pp:>4} {mb:>4} {gpipe:>10.1%} {ofob:>10.1%} {interleaved:>12.1%}')
    print()

print('--- 1F1B Peak Memory Analysis ---')
pp = 4
micro_batch = 8
activation_per_mb = 2.0
gpipe_peak = micro_batch * activation_per_mb
onefoneb_peak = (1 + pp - 1) * activation_per_mb
print(f'PP={pp}, micro_batch={micro_batch}, activation_per_MB={activation_per_mb}GB')
print(f'GPipe peak activation memory: {gpipe_peak:.1f} GB (all {micro_batch} micro-batches)')
print(f'1F1B peak activation memory:  {onefoneb_peak:.1f} GB (1 + {pp-1} micro-batches)')
print(f'1F1B saves {(1- onefoneb_peak/gpipe_peak):.0%} activation memory vs GPipe')

print(f'\nKey: 1F1B reduces both bubble rate and peak memory compared to GPipe.')
print(f'Interleaved 1F1B further reduces bubbles but increases communication.')
print(f'More micro-batches = lower bubble rate but higher latency.')

## 3. 通信-计算重叠

**目的**：在计算的同时执行通信，隐藏通信延迟

**核心思想**：GPU的SM（流多处理器）可以同时执行计算kernel和通信kernel。通过将梯度分桶（bucket），在一个桶计算梯度时，同时通信另一个桶的梯度。

**重叠策略**：
- **DP梯度同步重叠**：反向传播时，梯度计算完一个bucket就立即AllReduce，不等所有梯度算完
- **TP通信重叠**：利用CUDA stream，在attention/FFN计算时同时执行AllReduce
- **PP通信重叠**：在等待前一stage数据时，执行当前stage的反向传播

**关键参数**：
- `bucket_cap_mb`（DDP）：梯度桶大小，太小通信次数多，太大等待时间长
- 典型值：25MB（PyTorch DDP默认值）

In [ ]:
import torch
import time

torch.manual_seed(42)

print('=== Communication-Computation Overlap ===')

n_layers = 32
compute_time_per_layer = 5.0
comm_time_per_layer = 2.0

no_overlap_time = n_layers * (compute_time_per_layer + comm_time_per_layer)
overlap_time = n_layers * compute_time_per_layer + comm_time_per_layer
overlap_ratio = 1 - overlap_time / no_overlap_time

print(f'Per-layer: compute={compute_time_per_layer}ms, communication={comm_time_per_layer}ms')
print(f'\nNo overlap: {n_layers} × ({compute_time_per_layer}+{comm_time_per_layer}) = {no_overlap_time:.0f}ms')
print(f'With overlap: {n_layers} × {compute_time_per_layer} + {comm_time_per_layer} = {overlap_time:.0f}ms')
print(f'Speedup: {no_overlap_time/overlap_time:.2f}x ({overlap_ratio:.0%} time saved)')

print(f'\n--- Bucket Size Impact (DP gradient sync) ---')
model_params_mb = 14000
bucket_sizes = [5, 10, 25, 50, 100, 500]
allreduce_latency = 0.2
allreduce_bandwidth = 12.0

print(f'{"Bucket(MB)":>12} {"#Buckets":>10} {"Latency(ms)":>12} {"Transfer(ms)":>14} {"Total(ms)":>12}')
for bs in bucket_sizes:
    n_buckets = max(1, model_params_mb // bs)
    latency = n_buckets * allreduce_latency
    transfer = model_params_mb / allreduce_bandwidth
    total = latency + transfer
    print(f'{bs:>12} {n_buckets:>10} {latency:>12.1f} {transfer:>14.1f} {total:>12.1f}')

print(f'\nKey: Overlapping communication with computation is critical for scaling efficiency.')
print(f'Smaller buckets = more overlap opportunity but more latency overhead.')
print(f'PyTorch DDP default bucket_cap_mb=25MB is a good starting point.')

## 4. 并行配置搜索与ZeRO结合

**目的**：为给定模型和集群找到最优并行配置

**搜索空间**：
- TP ∈ {1, 2, 4, 8}（受节点内GPU数限制）
- PP ∈ {1, 2, 4, 8, 16, ...}（受模型层数限制）
- DP = total_gpus / (TP × PP)
- ZeRO stage ∈ {0, 1, 2, 3}

**优化目标**：最大化TFLOPS/GPU（训练吞吐量）

**约束**：
- 每GPU显存 ≤ 80GB（A100-80GB）
- 全局batch size = micro_batch × DP × gradient_accumulation
- DP ≥ 1（至少1个数据并行副本）

**ZeRO与3D并行结合**：
- ZeRO-1 + 3D：优化器状态分片，减少30-40%显存
- ZeRO-2 + 3D：+梯度分片，减少50-60%显存
- ZeRO-3 + 3D：+参数分片，可训练任意大模型，但通信量大

In [ ]:
import torch

print('=== Parallel Config Search ===')

def estimate_throughput(tp, pp, dp, zero_stage, params_b, n_layers, d_model, seq_len, micro_bs):
    per_gpu_model = params_b * 2 / tp / pp
    if zero_stage >= 1:
        opt_mem = params_b * 12 / tp / pp / dp
    else:
        opt_mem = params_b * 12 / tp / pp
    if zero_stage >= 2:
        grad_mem = params_b * 2 / tp / pp / dp
    else:
        grad_mem = params_b * 2 / tp / pp
    if zero_stage >= 3:
        param_mem = params_b * 2 / tp / pp / dp
    else:
        param_mem = per_gpu_model
    act_mem = d_model * n_layers * 4 * 2 * seq_len * micro_bs / tp / 1e9
    total = param_mem + grad_mem + opt_mem + act_mem

    if total > 80:
        return None, total

    tp_comm_ratio = 2 * 2 / tp
    pp_bubble = (pp - 1) / (micro_bs * dp + pp - 1)
    dp_comm_ratio = 1.0 / dp if zero_stage < 3 else 3.0 / dp
    compute_eff = max(0.1, 1.0 - tp_comm_ratio * 0.1 - pp_bubble - dp_comm_ratio * 0.05)
    tflops = 6 * params_b * seq_len * micro_bs * dp / 1e6 * compute_eff
    return tflops, total

params_b = 70
n_layers = 80
d_model = 8192
seq_len = 4096
micro_bs = 1
total_gpus = 64

print(f'Model: {params_b}B params, {total_gpus} GPUs')
print(f'\n{"TP":>3} {"PP":>3} {"DP":>3} {"ZeRO":>5} {"GPU_MEM":>10} {"TFLOPS":>10}')
print('-' * 45)

best_config = None
best_tflops = 0
for tp in [1, 2, 4, 8]:
    for pp in [1, 2, 4, 8, 16]:
        dp = total_gpus // (tp * pp)
        if dp < 1 or tp * pp * dp != total_gpus:
            continue
        for zero in [0, 1, 2, 3]:
            tflops, mem = estimate_throughput(tp, pp, dp, zero, params_b, n_layers, d_model, seq_len, micro_bs)
            if tflops is not None:
                print(f'{tp:>3} {pp:>3} {dp:>3} {zero:>5} {mem:>9.1f}GB {tflops:>10.1f}')
                if tflops > best_tflops:
                    best_tflops = tflops
                    best_config = (tp, pp, dp, zero)

if best_config:
    tp, pp, dp, zero = best_config
    print(f'\nBest config: TP={tp}, PP={pp}, DP={dp}, ZeRO={zero} → {best_tflops:.1f} TFLOPS')

print(f'\nKey: Config search balances memory constraints with throughput.')
print(f'ZeRO-1/2 with 3D parallelism is the sweet spot for most large models.')
print(f'ZeRO-3 enables training arbitrarily large models but with communication overhead.')